# Heat Equation Inverse with DeepXDE
I attempt to solve an inverse problem of the Heat Equation with DeepXDE.

In [1]:
import deepxde as dde
import numpy as np
import torch

Using backend: pytorch
Other supported backends: tensorflow.compat.v1, tensorflow, jax, paddle.
paddle supports more examples now and is recommended.


In [3]:
# define the domain
geom = dde.geometry.Interval(0.0, 1.0) # x from 0 to 1
timedomain = dde.geometry.TimeDomain(0.0, 1.0) # t from 0 to 1
geomtime = dde.geometry.GeometryXTime(geom, timedomain) # x vs t

# 1D Wave Equation
d2u/dt2 = c2 d2u/dx2

In [13]:
# Define the PDE
c = dde.Variable(0.5) # wave speed, to be trained

def wave_pde(x, u):
    u_tt = dde.grad.hessian(u, x, i=0, j=1) # with respect to t
    u_xx = dde.grad.hessian(u, x, i=0, j=0) # with respect to x
    return u_tt - c**2 * u_xx

def initial_condition(x):
    return np.sin(np.pi * x[:, 0:1])  # u(x,0) = sin(pi*x)

ic = dde.icbc.IC(
    geomtime,
    initial_condition,
    lambda _, on_initial: on_initial
)

# boundary conditions
bc_left = dde.icbc.DirichletBC(
    geomtime, lambda x: 0.0, lambda x, on_boundary: on_boundary and np.isclose(x[0], 0.0)
)
bc_right = dde.icbc.DirichletBC(
    geomtime, lambda x: 0.0, lambda x, on_boundary: on_boundary and np.isclose(x[0], 1.0)
)

In [15]:
# generate observational data (will need real solution class)

c_true = 0.1

def true_solution(x):
    return np.sin(np.pi * x[:, 0:1]) * np.cos(c_true * np.pi * x[:, 1:2])

N = 100

# bunch of x and t points
x_obs = np.random.rand(N, 1)
t_obs = np.random.rand(N, 1)
# combine:
X_obs = np.hstack((x_obs, t_obs))

# solution at those points
u_obs = true_solution(X_obs) # don't need to split because it's already combined

# add some noise
noise_level = 0.01
u_obs += noise_level * np.random.randn(*u_obs.shape)

# final observational data
observe_u = dde.icbc.PointSetBC(X_obs, u_obs)

In [16]:
# data object
data = dde.data.TimePDE(
    geomtime,
    wave_pde,
    [ic, bc_left, bc_right, observe_u],
    num_domain=10000,
    num_boundary=200,
    num_initial=200
)

# neural network
net = dde.nn.FNN(
    [2] + [50] * 3 + [1],
    activation="tanh",
    kernel_initializer="Glorot normal"
)

In [17]:
model = dde.Model(data, net)

model.compile(
    optimizer="adam",
    lr=1e-3,
    external_trainable_variables=[c]
)

Compiling model...
'compile' took 0.000373 s



In [18]:
model.train(epochs=5000)

print("Learned c:", c.item())

Training model...

Step      Train loss                                            Test loss                                             Test metric
0         [5.60e-05, 3.56e-01, 2.21e-02, 1.20e-01, 2.46e-01]    [5.60e-05, 3.56e-01, 2.21e-02, 1.20e-01, 2.46e-01]    []  
1000      [5.63e-05, 1.75e-04, 4.10e-05, 1.39e-04, 4.59e-04]    [5.63e-05, 1.75e-04, 4.10e-05, 1.39e-04, 4.59e-04]    []  
2000      [1.50e-05, 5.82e-05, 1.63e-05, 4.03e-05, 2.43e-04]    [1.50e-05, 5.82e-05, 1.63e-05, 4.03e-05, 2.43e-04]    []  
3000      [1.03e-05, 3.57e-05, 1.41e-05, 2.41e-05, 1.90e-04]    [1.03e-05, 3.57e-05, 1.41e-05, 2.41e-05, 1.90e-04]    []  
4000      [9.84e-06, 2.81e-05, 1.44e-05, 1.95e-05, 1.71e-04]    [9.84e-06, 2.81e-05, 1.44e-05, 1.95e-05, 1.71e-04]    []  
5000      [8.17e-06, 2.45e-05, 1.48e-05, 1.80e-05, 1.66e-04]    [8.17e-06, 2.45e-05, 1.48e-05, 1.80e-05, 1.66e-04]    []  

Best model at step 5000:
  train loss: 2.31e-04
  test loss: 2.31e-04
  test metric: []

'train' took 270.722385